# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SDlel/ml-assignments/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where could bias or leakage enter? What would you ask before trusting it?*

Finding 1: pages with stronger observed search visibility are better candidates for refresh review. My question is whether this is mostly measuring demand, or whether high-visibility pages simply have more chances to show any movement.

Finding 2: stale or older content can be a useful refresh signal. My question is whether age is acting as a real content-quality proxy or just standing in for page type, client, or publishing workflow differences.

In [1]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "scripts").exists())

WORK_OUTPUTS = ROOT / "work" / "outputs"
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import display_path


def run_script(script_name):
    subprocess.run([sys.executable, str(ROOT / "scripts" / script_name)], cwd=ROOT, check=True)

for script in ["01_prepare_features.py", "02_baseline_score.py", "03_train_model.py", "04_evaluate_and_export.py"]:
    run_script(script)

model_results = json.loads((ROOT / "outputs" / "model_results.json").read_text())
summary = json.loads((ROOT / "outputs" / "summary.json").read_text())
feature_meta = json.loads((ROOT / "data" / "processed" / "feature_metadata.json").read_text())

pd.DataFrame([
    {"receipt": "rows scored", "value": summary["rows_scored"]},
    {"receipt": "split", "value": model_results["split_strategy"]},
    {"receipt": "best model", "value": model_results["best_model"]["name"]},
    {"receipt": "target positive rate", "value": round(feature_meta["declining_rate"], 3)},
])

,receipt,value
0,rows scored,30000
1,split,client_holdout
2,best model,random_forest
3,target positive rate,0.542


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show before/after if you changed it.*

The honest split used here is the client-holdout split from Week 5. I am not claiming causality, only that the model ranked observed declining pages better than the baseline on this measured holdout.

In [2]:
metrics = model_results["models"][model_results["best_model"]["name"]]
baseline = model_results["baseline"]
honest_split_table = pd.DataFrame([
    {"method": "baseline_rules", "split": model_results["split_strategy"], "precision_at_50": baseline["baseline_precision_at_50"], "roc_auc": baseline["baseline_roc_auc"], "average_precision": baseline["baseline_average_precision"]},
    {"method": model_results["best_model"]["name"], "split": model_results["split_strategy"], "precision_at_50": metrics["precision_at_50"], "roc_auc": metrics["roc_auc"], "average_precision": metrics["average_precision"]},
])
honest_split_table.to_csv(WORK_OUTPUTS / "validation_holdout_metrics.csv", index=False)
honest_split_table.round(3)

,method,split,precision_at_50,roc_auc,average_precision
0,baseline_rules,client_holdout,0.24,0.627,0.468
1,random_forest,client_holdout,0.68,0.747,0.610


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The final feature list excludes raw identifiers as model inputs, future windows, the label itself, and product-generated flags. Identifiers are kept only for joining outputs back to a review queue.

In [3]:
model_features = model_results["model_numeric_features"] + model_results["model_categorical_features"]
blocked_patterns = ["label", "future", "flag", "product", "rank", "score", "prob", "prediction", "after"]
leakage_audit = pd.DataFrame({
    "feature": model_features,
    "blocked_pattern_hit": [
        ",".join(term for term in blocked_patterns if term in feature.lower()) or "none"
        for feature in model_features
    ],
})
leakage_audit.to_csv(WORK_OUTPUTS / "feature_leakage_audit.csv", index=False)
leakage_audit["blocked_pattern_hit"].value_counts().rename_axis("result").reset_index(name="count")

,result,count
0,none,26


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Bold version: "The model finds the pages that need to be refreshed."

Safer version: "On the bundled anonymized sample, the model ranked observed declining pages ahead of the baseline on a client-holdout split, so it may be useful as decision support for choosing pages to manually review first."

In [4]:
claim_rewrite = pd.DataFrame([
    {
        "unsafe_claim": "The model finds the pages that need to be refreshed.",
        "safe_claim": "On the bundled anonymized sample, the model ranked observed declining pages ahead of the baseline on a client-holdout split, so it may support manual refresh prioritization.",
    }
])
claim_rewrite.to_csv(WORK_OUTPUTS / "claim_rewrite.csv", index=False)
claim_rewrite

,unsafe_claim,safe_claim
0,The model finds the pages that need to be refr...,"On the bundled anonymized sample, the model ra..."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled ? markdown thinking AND code where code is requested.
- [x] Every number comes from a query, dataframe, or saved output.
- [x] I used careful language: observed / measured / directional / decision-support.
- [x] I did not include private client names, domains, URLs, titles, or keywords.